# 07 - Visualizacao da Segmentacao Binarizada

Le as tabelas analiticas persistidas pelo notebook 06, reconstrói a base linha a linha em memoria para a execucao escolhida em `config.toml` e gera um relatorio PDF focado na comparacao entre estrategias de binarizacao usando `iou`, `precision` e `recall`.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
plt.rcParams['figure.max_open_warning'] = 0
import pandas as pd
from IPython.display import display

root_dir = Path.cwd()
if not (root_dir / 'src').exists() and (root_dir.parent / 'src').exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.analysis import build_binarized_metrics_dataframe
from src.config import SEGMENTACAO_BINARIZADA_ANALISE_EXECUCAO
from src.io.path_resolver import PathResolver
from src.repositories import (
    AnaliseSegmentacaoBinarizadaInteracaoTagEstrategiaRepository,
    AnaliseSegmentacaoBinarizadaIntervaloConfiancaRepository,
    AnaliseSegmentacaoBinarizadaResumoEstrategiaRepository,
    AnaliseSegmentacaoBinarizadaResumoModeloEstrategiaRepository,
    AnaliseSegmentacaoBinarizadaTesteEstrategiaRepository,
    ImagemRepository,
)
from src.visualization import (
    PdfReportSection,
    plot_metric_bars_by_model,
    plot_metric_correlation_heatmap,
    plot_metric_distribution_by_model,
    plot_metric_scatter,
    plot_model_tag_interaction_heatmap,
    plot_pairwise_pvalue_heatmap,
    plot_simple_regression,
    save_pdf_report,
)


## Carregamento das tabelas analiticas e da base linha a linha

A visualizacao usa a base linha a linha reconstruida a partir do SQLite e as tabelas analiticas persistidas pelo notebook 06 para a execucao configurada da segmentacao binarizada.


In [ ]:
def _registros_para_dataframe(registros, columns):
    return pd.DataFrame([{column: getattr(registro, column) for column in columns} for registro in registros], columns=columns)

metric_names = ['iou', 'precision', 'recall']
metric_pairs = [('iou', 'precision'), ('iou', 'recall'), ('precision', 'recall')]
interaction_metrics = ['iou', 'precision', 'recall']
path_resolver = PathResolver.from_config()
report_output_path = Path(path_resolver.generated_dir) / '07_visualizacao_segmentacao_binarizada.pdf'

imagem_repository = ImagemRepository()
# Docs: docs/decisoes-tecnicas/analise-da-segmentacao-binarizada-por-execucao-fixa.md
df_base = build_binarized_metrics_dataframe(
    imagem_repository.list(),
    execucao_escolhida=SEGMENTACAO_BINARIZADA_ANALISE_EXECUCAO,
)
df_base_por_estrategia = df_base.drop(columns=['modelo']).rename(columns={'estrategia_binarizacao': 'modelo'})

df_resumo_estrategia = _registros_para_dataframe(
    AnaliseSegmentacaoBinarizadaResumoEstrategiaRepository().list(),
    ['estrategia_binarizacao', 'metric_name', 'count', 'mean', 'median', 'std', 'min', 'max', 'q1', 'q3', 'iqr', 'higher_is_better'],
)
df_resumo_modelo_estrategia = _registros_para_dataframe(
    AnaliseSegmentacaoBinarizadaResumoModeloEstrategiaRepository().list(),
    ['nome_modelo', 'estrategia_binarizacao', 'metric_name', 'count', 'mean', 'median', 'std', 'min', 'max', 'q1', 'q3', 'iqr', 'higher_is_better'],
)
df_intervalo_confianca = _registros_para_dataframe(
    AnaliseSegmentacaoBinarizadaIntervaloConfiancaRepository().list(),
    ['nome_modelo', 'estrategia_binarizacao', 'metric_name', 'statistic_name', 'count', 'estimate', 'ci_low', 'ci_high', 'confidence_level', 'n_resamples', 'higher_is_better'],
)
df_testes_estrategia = _registros_para_dataframe(
    AnaliseSegmentacaoBinarizadaTesteEstrategiaRepository().list(),
    ['metric_name', 'modelo_origem', 'comparison_scope', 'test_name', 'group_a', 'group_b', 'n_group_a', 'n_group_b', 'statistic', 'p_value', 'p_value_adjusted', 'effect_size', 'effect_size_label', 'mean_group_a', 'mean_group_b', 'median_group_a', 'median_group_b', 'favored_group'],
)
df_interacoes_tag_estrategia = _registros_para_dataframe(
    AnaliseSegmentacaoBinarizadaInteracaoTagEstrategiaRepository().list(),
    ['estrategia_binarizacao', 'tag_name', 'metric_name', 'count_com_tag', 'count_sem_tag', 'mean_com_tag', 'mean_sem_tag', 'median_com_tag', 'median_sem_tag', 'delta_mean', 'delta_median', 'relative_delta_mean', 'adjusted_delta_mean', 'adjusted_delta_median', 'impact_direction', 'higher_is_better'],
)

print(f'Execucao escolhida para a analise binarizada: {SEGMENTACAO_BINARIZADA_ANALISE_EXECUCAO}')
print(f'Registros da base linha a linha: {len(df_base)}')
print(f'Resumo por estrategia: {len(df_resumo_estrategia)}')
print(f'Resumo por modelo + estrategia: {len(df_resumo_modelo_estrategia)}')
print(f'Intervalos de confianca: {len(df_intervalo_confianca)}')
print(f'Testes entre estrategias: {len(df_testes_estrategia)}')
print(f'Interacoes estrategia x tag: {len(df_interacoes_tag_estrategia)}')
print(f'PDF de saida: {report_output_path}')


## Comparacao geral entre estrategias

Este bloco resume cada metrica no nivel da estrategia de binarizacao. Ele responde primeiro a pergunta mais direta: qual estrategia tende a performar melhor no agregado, antes de abrir por modelo.


In [ ]:
texto_estrategias = 'Este bloco resume cada metrica no nivel da estrategia de binarizacao. Ele responde primeiro a pergunta mais direta: qual estrategia tende a performar melhor no agregado, antes de abrir por modelo.'
figures_estrategias = []

df_resumo_estrategia_plot = df_resumo_estrategia.rename(columns={'estrategia_binarizacao': 'nome_modelo'})
display(df_resumo_estrategia.pivot(index='estrategia_binarizacao', columns='metric_name', values='mean').sort_index())

for metric_name in metric_names:
    fig, _ = plot_metric_bars_by_model(df_resumo_estrategia_plot, metric_name)
    figures_estrategias.append(fig)
    plt.show()


## Distribuicoes por estrategia

Os boxplots ajudam a ver dispersao e outliers por estrategia, sem misturar isso com a identidade do modelo.


In [ ]:
texto_distribuicao = 'Os boxplots ajudam a ver dispersao e outliers por estrategia, sem misturar isso com a identidade do modelo.'
figures_distribuicao = []

for metric_name in metric_names:
    fig, _ = plot_metric_distribution_by_model(df_base_por_estrategia, metric_name)
    figures_distribuicao.append(fig)
    plt.show()


## Comparacao estatistica entre estrategias

Aqui entram os testes nao parametricos entre estrategias.


In [ ]:
texto_testes = 'Aqui entram os testes nao parametricos entre estrategias.'
figures_testes = []

df_testes_estrategia_global = df_testes_estrategia.loc[df_testes_estrategia['modelo_origem'] == '__global__'].copy()
if df_testes_estrategia_global.empty:
    print('Nao ha testes globais entre estrategias disponiveis no SQLite atual.')
else:
    display(df_testes_estrategia_global.sort_values(['metric_name', 'comparison_scope', 'p_value_adjusted']).head(30))

    for metric_name in metric_names:
        fig, _ = plot_pairwise_pvalue_heatmap(df_testes_estrategia_global, metric_name)
        figures_testes.append(fig)
        plt.show()


## Analise bivariada e correlacao

Aqui verificamos associacao entre `iou`, `precision` e `recall`.


In [ ]:
texto_correlacao = 'Aqui verificamos associacao entre iou, precision e recall.'
figures_correlacao = []

display(df_base[metric_names].corr(method='pearson'))
display(df_base[metric_names].corr(method='spearman'))

for x_metric, y_metric in metric_pairs:
    fig, _ = plot_metric_scatter(df_base_por_estrategia, x_metric, y_metric)
    figures_correlacao.append(fig)
    plt.show()

for method in ['pearson', 'spearman']:
    fig, _ = plot_metric_correlation_heatmap(df_base, metric_names, method)
    figures_correlacao.append(fig)
    plt.show()


## Regressao simples

A regressao simples usa `num_tags_problema` como proxy de dificuldade.


In [ ]:
texto_regressao = 'A regressao simples usa num_tags_problema como proxy de dificuldade.'
figures_regressao = []

for metric_name in metric_names:
    fig, _ = plot_simple_regression(df_base_por_estrategia, 'num_tags_problema', metric_name)
    figures_regressao.append(fig)
    plt.show()


## Interacao entre estrategia e dificuldade

Este bloco resume como cada tag de problema desloca as metricas de cada estrategia.


In [ ]:
texto_interacoes = 'Este bloco resume como cada tag de problema desloca as metricas de cada estrategia.'
figures_interacoes = []

df_interacoes_plot = df_interacoes_tag_estrategia.rename(columns={'estrategia_binarizacao': 'nome_modelo'})
display(df_interacoes_tag_estrategia.sort_values(['metric_name', 'tag_name', 'adjusted_delta_mean']).head(30))

for metric_name in interaction_metrics:
    fig, _ = plot_model_tag_interaction_heatmap(df_interacoes_plot, metric_name)
    figures_interacoes.append(fig)
    plt.show()


## Leitura inicial

O notebook 07 prioriza uma leitura objetiva da segmentacao binarizada: comparacao geral entre estrategias, distribuicoes, significancia estatistica entre estrategias, relacao entre metricas e sensibilidade a dificuldade.


In [ ]:
report_sections = [
    PdfReportSection(heading='Carregamento das tabelas analiticas e da base linha a linha', body='A visualizacao usa a base linha a linha reconstruida a partir do SQLite e as tabelas analiticas persistidas pelo notebook 06 para a execucao escolhida da segmentacao binarizada.'),
    PdfReportSection(heading='Comparacao geral entre estrategias', body=texto_estrategias, figures=figures_estrategias),
    PdfReportSection(heading='Distribuicoes por estrategia', body=texto_distribuicao, figures=figures_distribuicao),
    PdfReportSection(heading='Comparacao estatistica entre estrategias', body=texto_testes, figures=figures_testes),
    PdfReportSection(heading='Analise bivariada e correlacao', body=texto_correlacao, figures=figures_correlacao),
    PdfReportSection(heading='Regressao simples', body=texto_regressao, figures=figures_regressao),
    PdfReportSection(heading='Interacao entre estrategia e dificuldade', body=texto_interacoes, figures=figures_interacoes),
    PdfReportSection(heading='Leitura inicial', body='O notebook 07 prioriza uma leitura objetiva da segmentacao binarizada em uma execucao configurada: comparacao geral entre estrategias, distribuicoes, significancia estatistica entre estrategias, relacao entre metricas e sensibilidade a dificuldade.'),
]

pdf_path = save_pdf_report(
    output_path=report_output_path,
    sections=report_sections,
    report_title='07 - Visualizacao da Segmentacao Binarizada',
)

print(f'Relatorio PDF salvo em: {pdf_path}')
